# 02a — Groundsource: Pluvial Flood Detection (PFDI)

**Purpose:** Calculate the Pluvial Flood Detection Index (PFDI) for each flood event by extracting
upstream drainage area (UPA) statistics from the MERIT Hydro dataset via Google Earth Engine.
PFDI = UPA / polygon_area. A value < 1 indicates a pluvial (local-rainfall-driven) flood;
a value > 1 indicates a fluvial (river-driven) flood.

**Method:** Events are processed in batches of 50 via the GEE API. The script supports smart
resume — if interrupted, it automatically picks up from the last completed batch.

**Input:**
- `outputs/groundsource_urban_df.parquet` — flood events with urban metrics (from 01a)
- MERIT Hydro (GEE: `MERIT/Hydro/v1_0_1`) — 90 m upstream drainage area raster

**Output:**
- `outputs/pfdi_batches/pfdi_batch_*.parquet` — intermediate per-batch results
- `outputs/groundsource_df_with_pfdi.parquet` — all events merged with PFDI columns:
  - `upa_max`, `upa_p95`, `upa_p99` : UPA statistics within each polygon (km²)
  - `poly_area_km2`                  : polygon area in equal-area projection (km²)
  - `PFDI_p95`, `PFDI_p99`, `PFDI_max` : PFDI indices (UPA / area)

In [ ]:
import os
import glob
import time
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import ee
from shapely import wkb as shapely_wkb

warnings.filterwarnings('ignore')

In [ ]:
# --- CONFIGURATION ---

INPUT_PATH       = r"D:\MY_CODES\global_urban_flood_database\groundsource\outputs\groundsource_urban_df.parquet"
BATCH_DIR        = r"D:\MY_CODES\global_urban_flood_database\groundsource\outputs\pfdi_batches"
OUTPUT_PATH      = r"D:\MY_CODES\global_urban_flood_database\groundsource\outputs\groundsource_df_with_pfdi.parquet"

GEE_PROJECT      = 'your-gee-project'   # Replace with your GEE project ID
BATCH_SIZE       = 50                   # events per GEE request (GEE limit)
N_BATCHES_TO_RUN = 5_000                # batches to process in this session; re-run to continue
SCALE_M          = 90                   # MERIT Hydro pixel size in metres
TILE_SCALE       = 8                    # GEE tileScale for memory efficiency
EQUAL_AREA_CRS   = "EPSG:6933"          # for polygon area calculation
WGS84_CRS        = "EPSG:4326"

## 1. Initialise GEE and load data

In [ ]:
ee.Initialize(project=GEE_PROJECT)
print("Google Earth Engine initialised.")

merit = ee.Image("MERIT/Hydro/v1_0_1").select('upa')
print("MERIT Hydro image loaded.")

In [ ]:
os.makedirs(BATCH_DIR, exist_ok=True)

df = pd.read_parquet(INPUT_PATH)
df = df.reset_index(drop=True)
df['event_id'] = df.index   # stable integer index for resume logic
print(f"Loaded {len(df):,} records")

# Compute polygon area in equal-area projection (used for PFDI denominator)
t0 = time.time()
geoms_ea = gpd.GeoSeries.from_wkb(df['geometry'], crs=WGS84_CRS).to_crs(EQUAL_AREA_CRS)
df['poly_area_km2'] = geoms_ea.area / 1e6
print(f"Polygon areas computed in {time.time() - t0:.1f}s")
print(f"Area range: {df['poly_area_km2'].min():.4f} – {df['poly_area_km2'].max():.0f} km²")

## 2. Helper functions

In [ ]:
def fix_geometry_safe(geom):
    """Apply buffer(0) to fix minor geometry validity issues without altering shape."""
    try:
        if not geom.is_valid:
            return geom.buffer(0)
    except Exception:
        pass
    return geom


def shapely_to_ee(geom):
    """Convert a Shapely geometry to an Earth Engine Geometry object."""
    return ee.Geometry(geom.__geo_interface__)


def get_start_event_id(batch_dir):
    """
    Determine the next event_id to process based on already-completed batch files.

    Reads only the 'event_id' column from all batch parquet files to stay fast.
    Returns 0 if no batches have been completed yet.
    """
    files = glob.glob(os.path.join(batch_dir, 'pfdi_batch_*.parquet'))
    if not files:
        return 0
    dfs = [pd.read_parquet(f, columns=['event_id']) for f in files]
    return int(pd.concat(dfs)['event_id'].max()) + 1


def ee_fc_to_df(fc):
    """Convert an Earth Engine FeatureCollection to a pandas DataFrame."""
    try:
        features = fc.getInfo()['features']
    except Exception as e:
        print(f"  GEE error: {e}")
        return None
    rows = [f['properties'] for f in features]
    return pd.DataFrame(rows) if rows else None


def pick_stat_column(row, base, suffixes=('', '_1', '_2')):
    """
    Retrieve a statistic from a GEE result row, tolerating GEE's variable column naming.

    GEE sometimes appends '_1', '_2' to combined reducer outputs.
    """
    for s in suffixes:
        col = f"{base}{s}"
        if col in row and not pd.isna(row[col]):
            return float(row[col])
    return np.nan

## 3. Batch processing loop

Re-run this cell to continue from where the previous session stopped.
Set `N_BATCHES_TO_RUN` in the configuration block to control session length.

In [ ]:
start_id = get_start_event_id(BATCH_DIR)
subset   = df[df['event_id'] >= start_id].copy()
print(f"Resuming from event_id = {start_id}")
print(f"Events remaining: {len(subset):,}  /  total: {len(df):,}")

reducer = ee.Reducer.max().combine(
    ee.Reducer.percentile([95, 99]), sharedInputs=True
)

batches_run   = 0
t_session     = time.time()

for batch_start in range(0, len(subset), BATCH_SIZE):
    if batches_run >= N_BATCHES_TO_RUN:
        print(f"\nSession limit of {N_BATCHES_TO_RUN} batches reached. Re-run to continue.")
        break

    batch       = subset.iloc[batch_start : batch_start + BATCH_SIZE]
    id_min      = int(batch['event_id'].min())
    id_max      = int(batch['event_id'].max())
    out_file    = os.path.join(BATCH_DIR, f'pfdi_batch_{id_min}_{id_max}.parquet')

    if os.path.exists(out_file):
        batches_run += 1
        continue

    try:
        t0 = time.time()

        # Build Earth Engine FeatureCollection for this batch
        features = []
        for _, row in batch.iterrows():
            geom = shapely_wkb.loads(row['geometry'])
            geom = fix_geometry_safe(geom)
            feat = ee.Feature(shapely_to_ee(geom), {'event_id': int(row['event_id'])})
            features.append(feat)

        fc     = ee.FeatureCollection(features)
        result = merit.reduceRegions(
            collection=fc,
            reducer=reducer,
            scale=SCALE_M,
            tileScale=TILE_SCALE
        )

        result_df = ee_fc_to_df(result)
        if result_df is None or result_df.empty:
            batches_run += 1
            continue

        # Parse GEE output rows into clean records
        rows_out = []
        for _, r in result_df.iterrows():
            eid      = int(r.get('event_id', -1))
            upa_max  = pick_stat_column(r, 'max')
            upa_p95  = pick_stat_column(r, 'p95')
            upa_p99  = pick_stat_column(r, 'p99')
            src      = batch[batch['event_id'] == eid]
            poly_a   = float(src['poly_area_km2'].values[0]) if len(src) else np.nan

            rows_out.append({
                'event_id'    : eid,
                'upa_max'     : upa_max,
                'upa_p95'     : upa_p95,
                'upa_p99'     : upa_p99,
                'poly_area_km2': poly_a,
                'PFDI_p95'    : upa_p95 / poly_a if (poly_a and poly_a > 0) else np.nan,
                'PFDI_p99'    : upa_p99 / poly_a if (poly_a and poly_a > 0) else np.nan,
                'PFDI_max'    : upa_max / poly_a if (poly_a and poly_a > 0) else np.nan,
            })

        pd.DataFrame(rows_out).to_parquet(out_file, index=False)

        batches_run += 1
        pct = (start_id + batch_start + len(batch)) / len(df) * 100
        print(f"Batch {batches_run}/{N_BATCHES_TO_RUN}  "
              f"| events {id_min}–{id_max}  "
              f"| {time.time()-t0:.1f}s  "
              f"| overall {pct:.1f}%", end='\r')

    except Exception as e:
        print(f"\nBatch {id_min}–{id_max} failed: {e}")
        batches_run += 1
        continue

print(f"\nSession complete. Batches run: {batches_run}, elapsed: {time.time()-t_session:.0f}s")

## 4. Merge batches, quality control, and save

In [ ]:
batch_files = sorted(glob.glob(os.path.join(BATCH_DIR, 'pfdi_batch_*.parquet')))
print(f"Found {len(batch_files)} batch files")

pfdi_df = pd.concat([pd.read_parquet(f) for f in batch_files], ignore_index=True)
print(f"Total PFDI records before merge: {len(pfdi_df):,}")

# Join PFDI columns back to the full dataset on event_id
merged = df.merge(pfdi_df, on='event_id', how='inner')
print(f"After inner join: {len(merged):,}")

# Quality control: remove rows with missing or infinite PFDI values
key_cols = ['upa_p95', 'upa_p99', 'upa_max', 'poly_area_km2']
before_qc = len(merged)
merged = merged.dropna(subset=key_cols)
merged = merged[merged['poly_area_km2'] > 0]
merged = merged.replace([np.inf, -np.inf], np.nan).dropna(
    subset=['PFDI_p95', 'PFDI_p99', 'PFDI_max']
)
print(f"After QC: {len(merged):,}  (removed {before_qc - len(merged):,} rows)")

merged.to_parquet(OUTPUT_PATH, index=False)
print(f"Saved → {OUTPUT_PATH}")